In [24]:
import os
from glob import glob
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
import src
VIDEO_DIR = r"data/htvd/videos"
#get the current working directory
CWD = os.getcwd()
# Display the working directory
print("Current Working Directory:", CWD)
print("Video Directory:", VIDEO_DIR)

ModuleNotFoundError: No module named 'src'

**Video source + watching**

In [10]:
# Read video
video_name = 'cctv052x2004080516x01638.avi'  # video file name
video_path = os.path.join(VIDEO_DIR, video_name)  # build full path

video = cv2.VideoCapture(video_path)  # open video file

# Visualize video
while True:
    ret, frame = video.read()  # read one frame
    if not ret:                # stop if no frame is returned (end of video)
        break
    
    cv2.imshow('frame', frame)  # display current frame
    key = cv2.waitKey(5)              # wait ~5 ms
    if key == ord('q'):                     # stop if 'q' key is pressed
        break
        

video.release()                 # release video object
cv2.destroyAllWindows()         # close all OpenCV windows

**Read the video frames**

In [12]:
def read_video_segment(in_path, vid_seg = None):
    # read a video file and display all frames except the first one
    video = cv2.VideoCapture(in_path)  # open video file
    frames = []
    frame_idx = 0
    while True:
        ret, frame = video.read()  # read one frame
        if not ret:                # stop if no frame is returned (end of video)
            break
        if vid_seg is not None:
            if frame_idx < vid_seg[0]:
                frame_idx += 1
                continue
            if frame_idx > vid_seg[1]:
                break
        frames.append(frame)
        frame_idx += 1
    video.release()                 # release video object
    return frames

frames = read_video_segment(video_path)

**Mosaïque d'affichage des frames**

In [16]:
import cv2
import numpy as np
import math

TARGET_W, TARGET_H = 1920, 1080
n = len(frames)

cols = math.ceil(math.sqrt(n))
rows = math.ceil(n / cols)

cell_w = TARGET_W // cols
cell_h = TARGET_H // rows

mosaic = np.zeros((TARGET_H, TARGET_W, 3), dtype=np.uint8)

for i, frame in enumerate(frames):
    r = i // cols
    c = i % cols

    resized = cv2.resize(frame, (cell_w, cell_h))
    mosaic[
        r*cell_h:(r+1)*cell_h,
        c*cell_w:(c+1)*cell_w
    ] = resized

cv2.namedWindow("Mosaic", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Mosaic", TARGET_W, TARGET_H)
cv2.imshow("Mosaic", mosaic)
cv2.waitKey(0)
cv2.destroyAllWindows()


**Print the hitboxes**

In [ ]:
# Read video
video_name = 'cctv052x2004080516x01638.avi'  # video file name
video_path = os.path.join(VIDEO_DIR, video_name)  # build full path

video = cv2.VideoCapture(video_path)  # open video file

# Visualize video
while True:
    ret, frame = video.read()  # read one frame
    if not ret:                # stop if no frame is returned (end of video)
        break
    
    cv2.imshow('frame', frame)  # display current frame

    for (x, y, w, h) in face_rects:                         # Loop over detections (for each face)
    cv2.rectangle(face_img, (x, y), (x+w, y+h),
                    (255, 29, 0), 5)                     # Draw a rectangle around each vehicles



    key = cv2.waitKey(5)              # wait ~5 ms
    if key == ord('q'):                     # stop if 'q' key is pressed
        break
        

video.release()                 # release video object
cv2.destroyAllWindows()         # close all OpenCV windows

In [25]:
repertoire = "data/vd8c/labels"
repertoire_classes = os.path.join(repertoire, "classes.txt")

**Retrieve bounding boxes and class for each detected object**

In [26]:
with open(repertoire_classes, "r", encoding="utf-8") as f:
    classes = [line.strip() for line in f if line.strip()]

classe_dict = {i: classe for i, classe in enumerate(classes)}
print(classe_dict)

{0: 'auto', 1: 'bus', 2: 'car', 3: 'lcv', 4: 'motorcycle', 5: 'multiaxle', 6: 'tractor', 7: 'truck'}


In [5]:
import os
from PIL import Image, ImageDraw, ImageFont

IMG_DIR = "data/vd8c/images"
LBL_DIR = "data/vd8c/labels"

img_path = os.path.join(IMG_DIR, fname)
label_name = os.path.splitext(fname)[0] + ".txt"
txt_path = os.path.join(LBL_DIR, label_name)

img = Image.open(img_path).convert("RGB")
W, H = img.size
draw = ImageDraw.Draw(img)

with open(txt_path, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) != 5:
            continue

        class_id_str, x, y, w, h = parts
        x, y, w, h = map(float, (x, y, w, h))

        # YOLO -> coins en pixels
        x_min = (x - w / 2) * W
        y_min = (y - h / 2) * H
        x_max = (x + w / 2) * W
        y_max = (y + h / 2) * H

        # clamp dans l'image
        x_min = max(0, min(W - 1, x_min))
        y_min = max(0, min(H - 1, y_min))
        x_max = max(0, min(W - 1, x_max))
        y_max = max(0, min(H - 1, y_max))

        # rectangle
        draw.rectangle([x_min, y_min, x_max, y_max], outline="red", width=2)

        # texte = class_id (sans dictionnaire)
        label = class_id_str

        # position du texte (au-dessus si possible)
        tx, ty = x_min, max(0, y_min - 18)

        # fond du texte pour lisibilité
        text_bbox = draw.textbbox((tx, ty), label, font=font)
        draw.rectangle(text_bbox, fill="red")
        draw.text((tx, ty), label, fill="white", font=font)

img.show()
